# MCP Client Application (SSE)

In this tutorial, we build a LangGraph **agent that consumes tools from an MCP server over SSE (Server-Sent Events)**, instead of stdio (`08_LG_MCP_Client_Stdio.ipynb`) or streamable HTTP (`07_LG_MCP_Client_Http.ipynb`).

The MCP server (`MCP_Server/math_server_sse.py`) exposes one tool:

* `add(a, b)` — combines two numbers

We'll connect over the `sse` transport using the `mcp` SDK's client directly and hand the discovered tool to a LangGraph ReAct-style agent — the same graph shape used in `08_LG_MCP_Client_Stdio.ipynb`, just with a different transport connecting to the MCP server. (`langchain-mcp-adapters` doesn't yet support `mcp>=2.0.0` — see `mcp_tools.py` for the small bridge that wraps MCP tools as LangChain tools instead.)

**Before running this notebook**, start the MCP server in a separate terminal:

```bash
cd 02-LanggraphAdvanced/MCP_Server
pip install -r requirements.txt
python math_server_sse.py
```

It should print `MCP server running at http://127.0.0.1:8000/sse`.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_aws langchain_core langgraph langgraph-prebuilt "mcp>=2.0.0" load_dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = os.getenv('AWS_DEFAULT_REGION')

We'll use [LangSmith](https://docs.smith.langchain.com/) for [tracing](https://docs.smith.langchain.com/concepts/tracing).

In [ ]:
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial"

## Connect to the MCP server

For `sse` we give `sse_client` the server's SSE endpoint URL — `http://127.0.0.1:8000/sse` (matching the host/port set in `math_server_sse.py`) — instead of a `command` (stdio) or a `streamable_http` URL.

`load_mcp_tools` (in `mcp_tools.py`) opens the session, calls `list_tools`, and wraps every remote tool as a LangChain `StructuredTool` we can bind straight to a model — no manual JSON-RPC handling required.

In [ ]:
from contextlib import AsyncExitStack

from mcp import ClientSession
from mcp.client.sse import sse_client

from mcp_tools import load_mcp_tools

mcp_exit_stack = AsyncExitStack()
read_stream, write_stream = await mcp_exit_stack.enter_async_context(
    sse_client("http://127.0.0.1:8000/sse")
)
mcp_session = await mcp_exit_stack.enter_async_context(ClientSession(read_stream, write_stream))
await mcp_session.initialize()

tools = await load_mcp_tools(mcp_session)
for tool in tools:
    print(f"{tool.name}: {tool.description}")

## Bind the MCP tools to a model

Same pattern as the stdio and HTTP agents: bind the discovered tools to `ChatBedrockConverse` and let the model decide when to call them.

In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model_id="amazon.nova-pro-v1:0",
    temperature=0,
    max_tokens=None,
)
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

sys_msg = SystemMessage(
    content="You are a helpful assistant. Use the available MCP tools whenever they can answer the question."
)

# Node
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

## Build the graph

As before: an `assistant` node (the model with tools bound) and a `tools` node (a `ToolNode` that executes whichever tool call the model requested). `tools_condition` routes to `tools` when the model asked for a tool call, and to `END` otherwise; the edge from `tools` back to `assistant` lets the model use the tool result to keep reasoning or produce a final answer.

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
mcp_graph = builder.compile(checkpointer=memory)

try:
    display(Image(mcp_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(mcp_graph.get_graph().draw_ascii())

## Try it out

Each call below routes through `assistant` → `tools` (an MCP tool call over SSE) → `assistant` again for the final answer.

> Note: like `math_server.py` in the stdio example, `math_server_sse.py`'s `add` tool currently multiplies its two inputs — the response below reflects that, not a bug in this notebook.

In [ ]:
config = {"configurable": {"thread_id": "mcp-sse-demo-1"}}

result = await mcp_graph.ainvoke(
    {"messages": [HumanMessage(content="Use the add tool on 3 and 4, then tell me the result.")]},
    config=config,
)
for m in result["messages"]:
    m.pretty_print()

## Interactive chat loop

The graph is async because the MCP tool calls go over SSE (async HTTP streaming), so the interactive loop uses `ainvoke` instead of `invoke`.

In [ ]:
async def stream_graph_updates(user_input: str):
    result = await mcp_graph.ainvoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )
    print("Assistant:", result["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        await stream_graph_updates(user_input)
    except Exception:
        # fallback if input() is not available (e.g. non-interactive run)
        user_input = "Use the add tool on 10 and 15, then tell me the result."
        print("User: " + user_input)
        await stream_graph_updates(user_input)
        break